<a href="https://colab.research.google.com/github/Pallavibt/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pallavibt/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will use a supervised classification model to estimate which content items may need a refresh, based on the target label provided in the assignment. I will start with a simple Logistic Regression model because it is interpretable and provides a useful probability-based ranking of content items. I will compare it with my Week-4 baseline using the same data split and evaluation metric. The model will support content-review decisions; it will not prove that refreshing a page improves its performance.

In [11]:
import pandas as pd

url = "https://raw.githubusercontent.com/Pallavibt/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())


Dataset shape: (30000, 44)

Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [12]:
# Check available trend labels
print("Trend direction values:")
print(df["trend_direction"].value_counts(dropna=False))

print("\nTrend percentage summary:")
print(df["trend_pct"].describe())

print("\nMissing values in key columns:")
print(df[
    ["client_id", "trend_direction", "trend_pct",
     "impressions_90d", "impressions_last_30d",
     "impressions_prev_30d"]
].isna().sum())

Trend direction values:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Trend percentage summary:
count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64

Missing values in key columns:
client_id                  0
trend_direction            0
trend_pct               3388
impressions_90d            0
impressions_last_30d       0
impressions_prev_30d       0
dtype: int64


In [13]:
print("Baseline score file available in this session?")
import os

for path in [
    "work/outputs/baseline_action_score.csv",
    "baseline_action_score.csv"
]:
    print(path, "->", os.path.exists(path))

Baseline score file available in this session?
work/outputs/baseline_action_score.csv -> False
baseline_action_score.csv -> False


In [14]:

print("Dataset shape:", df.shape)

print("\nTrend direction counts:")
print(df["trend_direction"].value_counts(dropna=False))

print("\nBaseline-related columns:")
print([
    col for col in df.columns
    if "baseline" in col.lower()
    or "score" in col.lower()
    or "label" in col.lower()
])


Dataset shape: (30000, 44)

Trend direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Baseline-related columns:
[]


In [15]:

# Check data types and missing values before modeling
print("Data types:")
print(df.dtypes.to_string())

print("\nMissing values in columns:")
missing = df.isna().sum()
print(missing[missing > 0].sort_values(ascending=False).to_string())

print("\nUnique clients:", df["client_id"].nunique())


Data types:
content_id                 object
client_id                  object
search_volume             float64
competition               float64
competition_level          object
cpc                       float64
content_type               object
main_intent                object
word_count                float64
char_count                float64
provider_used              object
model_used                 object
impressions_90d             int64
clicks_90d                  int64
pageviews_90d               int64
sessions_90d                int64
users_90d                   int64
engaged_sessions_90d        int64
ai_sessions_90d             int64
scroll_events_90d           int64
days_with_impressions       int64
days_with_sessions          int64
impressions_last_30d        int64
clicks_last_30d             int64
sessions_last_30d           int64
impressions_prev_30d        int64
clicks_prev_30d             int64
sessions_prev_30d           int64
content_age_days            int64
ag

In [16]:

import pandas as pd

# Define a binary target from the observed trend category
model_df = df.copy()
model_df["target_declining"] = (
    model_df["trend_direction"] == "down"
).astype(int)

# Remove identifiers and trend-derived fields from model inputs
excluded = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "target_declining",
]

X = model_df.drop(columns=excluded)
y = model_df["target_declining"]
groups = model_df["client_id"]

print("Feature rows and columns:", X.shape)
print("\nTarget counts:")
print(y.value_counts().rename(index={0: "Not down", 1: "Down"}))
print("\nClient groups:", groups.nunique())
print("\nTarget prevalence:", round(y.mean() * 100, 2), "%")


Feature rows and columns: (30000, 40)

Target counts:
target_declining
Down        16262
Not down    13738
Name: count, dtype: int64

Client groups: 32

Target prevalence: 54.21 %


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a client-grouped train/test split, with approximately 80% of records for training and 20% for testing. Records from the same client are kept in only one split to reduce client-specific leakage. This tests generalization to clients not seen during training. The split is not time-aware, and the target represents an observed trend category rather than a future outcome, so the results should be interpreted as classification of observed trends.

In [17]:

from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))



Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I evaluated Logistic Regression and the reconstructed ML-07 rule-based baseline on the same client-grouped test set using Precision@20. Both methods achieved 1.00 Precision@20, meaning all 20 top-ranked records from each method belonged to the observed downward-trend class. Logistic Regression achieved 0.739 accuracy and 0.744 F1-score on the full test set. The top-20 metric alone does not establish that one method is better, and the baseline rules must be checked against the original ML-07 implementation. The target represents an observed trend, not a future outcome or proof that refreshing content improves performance.

In [18]:

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Identify numerical and categorical columns
numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = X_train.select_dtypes(exclude="number").columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

results = pd.DataFrame([{
    "model": "Logistic Regression",
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0)
}])

display(results.round(3))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(
    y_test, y_pred,
    target_names=["Not down", "Down"],
    zero_division=0
))



,model,accuracy,precision,recall,f1
0,Logistic Regression,0.739,0.745,0.742,0.744


Confusion matrix:
[[2214  800]
 [ 811 2338]]

Classification report:
              precision    recall  f1-score   support

    Not down       0.73      0.73      0.73      3014
        Down       0.75      0.74      0.74      3149

    accuracy                           0.74      6163
   macro avg       0.74      0.74      0.74      6163
weighted avg       0.74      0.74      0.74      6163



In [19]:

# Compare both rankings on the same held-out test records
test_df = df.iloc[test_idx].copy()
test_df["target_declining"] = y_test.to_numpy()

# Match the ML-07 rule-based baseline
test_df["avg_position_clean"] = test_df["avg_position"].replace(0, np.nan)

prev = test_df["impressions_prev_30d"]
last = test_df["impressions_last_30d"]

change_pct = ((last - prev) / prev.replace(0, np.nan)) * 100

test_df["baseline_score"] = (
    (test_df["impressions_90d"] > 0).astype(int)
    + 2 * (test_df["content_age_days"] >= 180).astype(int)
    + 2 * (test_df["days_since_last_update"] >= 180).astype(int)
    + (last > 0).astype(int)
    + 2 * test_df["avg_position_clean"].between(5, 30).astype(int)
    + 2 * ((prev >= 100) & (change_pct <= -20)).astype(int)
    + ((last >= 100) & test_df["ctr"].notna()
       & (test_df["ctr"] <= 1)).astype(int)
)

test_df["model_probability"] = model.predict_proba(X_test)[:, 1]

k = 20

baseline_top = test_df.sort_values(
    ["baseline_score", "impressions_90d", "content_id"],
    ascending=[False, False, True]
).head(k)

model_top = test_df.sort_values(
    ["model_probability", "content_id"],
    ascending=[False, True]
).head(k)

comparison = pd.DataFrame([
    {
        "method": "ML-07 baseline",
        "precision_at_20": baseline_top["target_declining"].mean()
    },
    {
        "method": "Logistic Regression",
        "precision_at_20": model_top["target_declining"].mean()
    }
])

display(comparison.round(3))


,method,precision_at_20
0,ML-07 baseline,1.0
1,Logistic Regression,1.0


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model produced 800 false positives and 811 false negatives on the held-out test set. Some false positives had stable or increasing trends, while some false negatives had very low recent impressions. This indicates that traffic levels and other content attributes do not perfectly distinguish the observed trend categories. The largest positive coefficients were associated with previous-period impressions and total impressions, while recent impressions had the largest negative coefficient. These are model associations, not causal effects. Results should be interpreted cautiously because the target is derived from an already-observed trend, and the experiment does not establish whether refreshing content improves performance.

In [20]:

# Inspect prediction errors
error_df = test_df.copy()
error_df["actual"] = y_test.to_numpy()
error_df["predicted"] = y_pred

false_positives = error_df[
    (error_df["actual"] == 0) & (error_df["predicted"] == 1)
]
false_negatives = error_df[
    (error_df["actual"] == 1) & (error_df["predicted"] == 0)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nFalse-positive examples:")
display(false_positives[
    ["content_type", "content_age_days", "impressions_90d",
     "impressions_prev_30d", "impressions_last_30d",
     "trend_direction"]
].head(5))

print("\nFalse-negative examples:")
display(false_negatives[
    ["content_type", "content_age_days", "impressions_90d",
     "impressions_prev_30d", "impressions_last_30d",
     "trend_direction"]
].head(5))

# Top model coefficients after preprocessing
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["classifier"].coef_[0]

importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

print("\nFeatures associated with the Down prediction:")
display(
    importance.sort_values("coefficient", ascending=False).head(10)
)

print("\nFeatures associated with the Not down prediction:")
display(
    importance.sort_values("coefficient", ascending=True).head(10)
)



False positives: 800
False negatives: 811

False-positive examples:


,content_type,content_age_days,impressions_90d,impressions_prev_30d,impressions_last_30d,trend_direction
13,keyword article,238,307,77,85,stable
36,keyword article,187,371,95,77,stable
64,keyword article,106,2639,250,363,up
78,keyword article,174,59,1,15,up
126,keyword article,126,82,11,25,up



False-negative examples:


,content_type,content_age_days,impressions_90d,impressions_prev_30d,impressions_last_30d,trend_direction
23,keyword article,502,297,34,0,down
25,keyword article,180,27,14,9,down
39,keyword article,348,4,4,0,down
43,keyword article,138,184,4,3,down
47,keyword article,126,8,6,1,down



Features associated with the Down prediction:


,feature,coefficient
18,numeric__impressions_prev_30d,29.535083
5,numeric__impressions_90d,1.921298
19,numeric__clicks_prev_30d,1.058300
8,numeric__sessions_90d,0.546320
13,numeric__days_with_impressions,0.539104
44,categorical__model_used_gpt-5-mini,0.436186
7,numeric__pageviews_90d,0.416507
69,categorical__position_tier_striking,0.364171
34,categorical__content_type_keyword article,0.329594
66,categorical__position_tier_deep,0.311555



Features associated with the Not down prediction:


,feature,coefficient
15,numeric__impressions_last_30d,-35.873439
70,categorical__position_tier_top_3,-1.253448
16,numeric__clicks_last_30d,-0.985563
37,categorical__main_intent_navigational,-0.547696
9,numeric__users_90d,-0.538767
21,numeric__content_age_days,-0.465682
17,numeric__sessions_last_30d,-0.465110
51,categorical__freshness_tier_181+,-0.429594
33,categorical__content_type_feedly article,-0.400708
41,categorical__model_used_gemini-2.5-flash,-0.379081


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.